# 강화학습 2강 — 탐험과 활용 (CartPole)

1강에서는 **무작위로만** 움직였습니다. 매번 처음 보는 것처럼 행동하니 아무것도 배우지 못했죠.

오늘은 두 가지를 섞습니다.
- **탐험(Exploration)** — 새로운 행동을 해 보며 경험을 쌓는다
- **활용(Exploitation)** — 지금까지의 경험으로 가장 이익이 되는 행동만 고른다

얼마나 탐험할지는 **엡실론(ε)** 하나로 정합니다. ε = 1 이면 전부 탐험, ε = 0 이면 전부 활용입니다.

> 위에서부터 순서대로 ▶ 실행하세요. (메뉴 「런타임 → 모두 실행」도 됩니다)

## 1. 상태를 칸으로 나누기

In [ ]:
import gymnasium as gym
import numpy as np
import random

# CartPole 의 상태는 숫자 4개입니다: [수레 위치, 수레 속도, 막대 각도, 막대 회전 속도]
# 숫자가 너무 잘게 나뉘어 있어서, 비슷한 상태끼리 「칸」으로 묶어 줍니다.
위치_경계 = [-0.8, 0.8]                        # 왼쪽 / 가운데 / 오른쪽  → 3칸
속도_경계 = [-0.5, 0.5]                        # 왼쪽으로 / 거의 멈춤 / 오른쪽으로 → 3칸
각도_경계 = np.linspace(-0.21, 0.21, 7)[1:-1]   # 막대가 얼마나 기울었나 → 6칸
회전_경계 = np.linspace(-2.0, 2.0, 7)[1:-1]     # 막대가 얼마나 빨리 넘어가나 → 6칸

def 칸_찾기(state):
    """상태(숫자 4개) → 칸 번호 4개"""
    return (
        int(np.digitize(state[0], 위치_경계)),
        int(np.digitize(state[1], 속도_경계)),
        int(np.digitize(state[2], 각도_경계)),
        int(np.digitize(state[3], 회전_경계)),
    )

env = gym.make("CartPole-v1")
state, info = env.reset()
print("상태(숫자 4개):", state)
print("칸 번호:", 칸_찾기(state))
env.close()

## 2. 점수표(Q표) · 행동 고르기 · 점수표 고치기

칸마다 「왼쪽으로 밀면 몇 점, 오른쪽으로 밀면 몇 점」을 적어 두는 표입니다. 이 점수를 **행동가치(Q)** 라고 부릅니다.
4강에서는 이 표를 딥러닝으로 바꾸고, 그게 **DQN** 입니다.

In [ ]:
# 점수표(Q표) 만들기 — 칸마다 [왼쪽으로 밀기 점수, 오른쪽으로 밀기 점수]
def 새_점수표():
    return np.zeros((3, 3, 6, 6, 2))

# 행동 고르기 — 엡실론(ε) 확률로 「탐험」, 나머지는 「활용」
def 행동_고르기(점수표, 칸, 엡실론):
    if random.random() < 엡실론:
        return random.randint(0, 1)          # 탐험: 아무거나 해 본다 (1강의 무작위 선택)
    return int(np.argmax(점수표[칸]))         # 활용: 지금까지 점수가 가장 높은 행동

# 점수표 고치기 — 「방금 받은 보상 + 다음 칸에서 기대되는 점수」 쪽으로 조금씩 옮긴다
배우는_속도 = 0.1    # 한 번에 얼마나 고칠까
미래_할인 = 0.99     # 미래 점수를 얼마나 믿을까

def 점수표_고치기(점수표, 칸, 행동, 보상, 다음_칸, 끝남):
    목표 = 보상 if 끝남 else 보상 + 미래_할인 * np.max(점수표[다음_칸])
    점수표[칸 + (행동,)] += 배우는_속도 * (목표 - 점수표[칸 + (행동,)])

점수표 = 새_점수표()
print("점수표 크기:", 점수표.shape, "→ 칸", 3*3*6*6, "개 × 행동 2개")

## 3. 플레이하면서 배우기

In [ ]:
def 학습하기(방식, 판수=1000):
    """방식: '탐험만' / '활용만' / '탐험 → 활용'"""
    env = gym.make("CartPole-v1")
    점수표 = 새_점수표()
    엡실론 = 1.0
    점수들 = []

    for 판 in range(1, 판수 + 1):
        if 방식 == "탐험만":
            엡실론 = 1.0                      # 언제나 무작위 (1강과 같다)
        elif 방식 == "활용만":
            엡실론 = 0.0                      # 한 번도 탐험하지 않는다

        state, info = env.reset()
        칸 = 칸_찾기(state)
        끝 = False
        점수 = 0
        while not 끝:
            행동 = 행동_고르기(점수표, 칸, 엡실론)
            next_state, reward, terminated, truncated, info = env.step(행동)
            끝 = terminated or truncated
            다음_칸 = 칸_찾기(next_state)
            점수표_고치기(점수표, 칸, 행동, reward, 다음_칸, terminated)
            칸 = 다음_칸
            점수 += reward
        점수들.append(점수)

        if 방식 == "탐험 → 활용":
            엡실론 = max(0.01, 엡실론 * 0.995)  # 판이 지날수록 탐험을 줄인다

        if 판 % 100 == 0:
            print(f"[{방식}] {판}판 · 탐험 비율 {엡실론:.2f} · 최근 100판 평균 {np.mean(점수들[-100:]):.0f}점")
    env.close()
    return 점수들, 점수표

## 4. 세 가지 방법 비교

In [ ]:
결과 = {}
for 방식 in ["탐험만", "활용만", "탐험 → 활용"]:
    점수들, 점수표 = 학습하기(방식)
    결과[방식] = (점수들, 점수표)
    print()

print("===== 마지막 100판 평균 =====")
for 방식, (점수들, _) in 결과.items():
    print(f"{방식:8s} : {np.mean(점수들[-100:]):6.1f}점")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "DejaVu Sans"
이름 = {"탐험만": "explore only (e=1)", "활용만": "exploit only (e=0)", "탐험 → 활용": "explore -> exploit"}

plt.figure(figsize=(10, 4))
for 방식, (점수들, _) in 결과.items():
    이동평균 = np.convolve(점수들, np.ones(20) / 20, mode="valid")
    plt.plot(이동평균, label=이름[방식])
plt.xlabel("episode")
plt.ylabel("score (20-episode average)")
plt.legend()
plt.show()

## 5. 다 배운 두뇌로 플레이 (활용만)

「탐험 → 활용」으로 배운 점수표를 들고, 이제는 탐험 없이 가장 좋은 행동만 고릅니다.

In [ ]:
점수표 = 결과["탐험 → 활용"][1]
env = gym.make("CartPole-v1")
for 판 in range(1, 6):
    state, info = env.reset()
    끝 = False
    점수 = 0
    while not 끝:
        행동 = 행동_고르기(점수표, 칸_찾기(state), 엡실론=0)
        state, reward, terminated, truncated, info = env.step(행동)
        끝 = terminated or truncated
        점수 += reward
    print(f"{판}판: {점수:.0f}점")
env.close()

## 정리

| 방법 | 결과 |
|---|---|
| 탐험만 (ε=1) | 1강과 같다. 경험을 쌓아도 쓰지 않으니 늘 20점 근처 |
| 활용만 (ε=0) | 처음 고른 행동만 계속 한다. 더 나은 길을 찾지 못하고 10점 근처에 갇힌다 |
| **탐험 → 활용** | 처음엔 많이 해 보고, 점점 아는 것을 쓴다. 점수가 크게 오른다 |

**다음 3강** — 지금은 경험을 한 번 쓰고 버립니다. 경험을 **기억 상자(리플레이 메모리)** 에 모아 두고 다시 꺼내 배우면 어떻게 될까요?